# geo-data-discovery — run it in Google Colab

**Juliano Ramanantsoa** · University of Bergen / Bjerknes Centre · MIT License

This notebook runs my data-discovery method end to end. I wrote it so anyone can
open it, press *Runtime → Run all*, and get **real, verified data** — not a
description of data. Colab has open internet, so every link here is resolved live;
if a dataset is reachable, you get it.

The rule the whole method rests on: a link is never shown unless it was resolved and
its content checked this run. You trust a status code you can re-run, not a model's memory.


## 1. Get the tool
Clone the repository and step into it. (Requires that the repo is on GitHub — see `DEPLOY.md`.)


In [ ]:
!git clone -q https://github.com/Juliano1111-ai/geo-data-discovery.git
%cd geo-data-discovery


## 2. Install the optional extras
The core engine needs nothing beyond the Python standard library. These packages add
waveform download/plotting and tables for the examples below.


In [ ]:
!pip install -q -r requirements.txt


## 3. Example A — raw seismic waveforms (Mount Etna, 2024)
A *waveform* request routes to the FDSN `dataselect` service (network `IV`, INGV). The
engine builds the endpoints, resolves them, validates the response, and splits the output.


In [ ]:
!python skill/geo-data-discovery/scripts/discovery_orchestrator.py \
  --query "Seismic waveforms Etna 2024 raw data" \
  --bbox "14.8,37.6,15.2,37.9" --start 2024-01-01 --end 2024-12-31 \
  --network IV --outdir out_etna


In [ ]:
import json
cat = json.load(open("out_etna/verified_catalog.jsonld"))
print(f"{len(cat)} verified record(s). Final status:", json.load(open("out_etna/workflow_status.json"))["final_status"])
print(json.dumps(cat, indent=2)[:1600])


## 4. Example B — earthquake catalogue (Himalaya, M ≥ 8, 1500–2026)
A *magnitude / event* request routes to earthquake **catalogues** (FDSN `event` service,
ISC-GEM, NOAA NCEI) — not waveforms. Here we pull the USGS ComCat event list straight
into a table.


In [ ]:
!python skill/geo-data-discovery/scripts/discovery_orchestrator.py \
  --query "Seismic event Himalaya 1500-2026 Magnitude 8" \
  --bbox "73,26,97,37" --start 1500-01-01 --end 2026-12-31 --outdir out_himalaya


In [ ]:
import pandas as pd
url = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
       "&starttime=1500-01-01&endtime=2026-12-31&minmagnitude=8"
       "&minlatitude=26&maxlatitude=37&minlongitude=73&maxlongitude=97&orderby=magnitude")
df = pd.read_csv(url)
print(len(df), "events of M>=8 in the Himalayan box (USGS ComCat)")
df[["time","place","mag","magType","depth"]].head(20)


## 5. Example C — download and plot a real waveform (ObsPy)
The production way to pull waveforms. This fetches one hour from summit station **ESLN**
(Serra La Nave) and plots it.


In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
cl = Client("http://webservices.ingv.it")
t0 = UTCDateTime("2024-01-01T00:00:00")
st = cl.get_waveforms("IV", "ESLN", "*", "HH?", t0, t0 + 3600)
print(st)
st.plot();


## 6. Use it for your own target
Change the four things that define any request and re-run cell 3 or 4:

- **query** — what you want (use *waveform/raw* for waveforms, *event/magnitude* for catalogues)
- **bbox** — `W,S,E,N` in decimal degrees
- **start / end** — ISO dates
- **network** — FDSN code for waveforms (e.g. `IV` INGV, `GE` GEOFON)

Outputs land in `out_*/`: `verified_catalog.jsonld` (what you can download),
`restricted_leads.json`, `unresolved_leads.json`, `access_validation_log.csv`, and
`workflow_status.json`. Full method and docs: `README.md`, `USAGE.md`,
`docs/finding_geoscience_data.md`.
